# 06. 장기 메모리 & 스킬

## 학습 목표
- `CompositeBackend` + `StoreBackend`로 장기 메모리를 구현한다
- 크로스 스레드 메모리 공유 패턴을 이해한다
- `AGENTS.md`를 통해 에이전트에 컨텍스트를 주입한다
- 스킬(SKILL.md)의 구조와 Progressive Disclosure를 이해한다
- Skills vs Memory의 차이를 파악한다

In [1]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

환경 설정 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


In [3]:
# OpenAI gpt-4.1 모델 설정
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

---
## 1. 장기 메모리가 필요한 이유

기본 `StateBackend` 에이전트는 **대화 스레드가 끝나면 모든 정보를 잊습니다**.  
하지만 실제 어시스턴트는 아래 정보를 **대화 간에 유지**해야 합니다:

- 사용자 선호도 (코딩 스타일, 사용 언어)
- 프로젝트 컨벤션 (아키텍처 결정, 네이밍 규칙)
- 이전 대화에서 학습한 피드백
- 자주 참조하는 정보 (API 문서, 설정값)

### 해결 방식: CompositeBackend

![CompositeBackend 라우팅](../../assets/images/composite_backend.png)

`/memories/` 경로에 저장된 파일은 **어떤 대화 스레드에서든** 접근할 수 있습니다.

In [4]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend, FilesystemBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver


# 1. 스토어와 체크포인터 생성
store = InMemoryStore()          # 개발용 (프로덕션: PostgresStore)
checkpointer = MemorySaver()     # 에이전트 상태 유지


# 2. CompositeBackend 팩토리 — /memories/만 영속, 나머지는 에페메럴
def memory_backend_factory(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/memories/": StoreBackend(runtime),
        },
    )


# 3. 에이전트 생성
memory_agent = create_deep_agent(
    model=model,
    system_prompt="""당신은 개인 어시스턴트입니다.
사용자가 기억해 달라고 하는 정보는 /memories/ 폴더에 저장하세요.
이전에 저장한 메모리가 있으면 참고하여 응답하세요.
한국어로 응답하세요.""",
    backend=memory_backend_factory,
    store=store,
    checkpointer=checkpointer,
)

print("장기 메모리 에이전트 생성 완료")

장기 메모리 에이전트 생성 완료


---
## 2. 크로스 스레드 메모리 공유

`StoreBackend`에 저장된 데이터는 **스레드 간에 공유**됩니다.  
아래 예제에서 스레드 1에서 저장한 선호도를 스레드 2에서 읽어봅니다.

In [5]:
# 스레드 1: 사용자 선호도 저장
config_t1 = {"configurable": {"thread_id": "memory-thread-1"}}

result1 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": """
다음 정보를 기억해 주세요:
- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터, 타입 힌트 필수
/memories/preferences.txt에 저장해 주세요.
"""}]},
    config={**config_t1, **lf_config},
)

print("[스레드 1] 저장 결과:")
print(result1["messages"][-1].content)

[스레드 1] 저장 결과:
다음 정보를 /memories/preferences.txt에 저장했습니다:
- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터, 타입 힌트 필수

필요할 때 이 정보를 참고하겠습니다.


In [6]:
# 스레드 2: 다른 대화에서 저장된 메모리 활용
config_t2 = {"configurable": {"thread_id": "memory-thread-2"}}

result2 = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "/memories/preferences.txt를 읽어서 내 선호도를 알려 주세요."}]},
    config={**config_t2, **lf_config},
)

print("[스레드 2] 메모리 읽기 결과:")
print(result2["messages"][-1].content)

[스레드 2] 메모리 읽기 결과:
당신의 선호도는 다음과 같습니다.

- 선호하는 프로그래밍 언어: Python
- 선호하는 에디터: VS Code
- 코딩 스타일: Black 포맷터 사용, 타입 힌트 필수

필요한 경우 이 기준에 맞춰 안내드리겠습니다.


---
## 3. AGENTS.md를 통한 컨텍스트 주입

`memory` 파라미터를 사용하면, 에이전트가 시작될 때 **AGENTS.md 파일을 자동으로 로드**하여  
시스템 프롬프트에 주입합니다.

### AGENTS.md란?
에이전트에게 항상 적용되어야 하는 **규칙, 컨벤션, 컨텍스트 정보**를 담는 마크다운 파일입니다.

### 특징
- 에이전트가 시작할 때 **항상 로드** (on-demand가 아님)
- `<agent_memory>` 태그로 시스템 프롬프트에 주입
- 여러 소스 지정 가능 (배열)
- 에이전트가 `edit_file` 도구로 AGENTS.md를 **스스로 업데이트** 가능

In [7]:
import tempfile

# 임시 디렉토리 생성 — FilesystemBackend의 root_dir로 사용
tmp_dir = tempfile.mkdtemp()
print(f"임시 디렉토리 생성: {tmp_dir}")

임시 디렉토리 생성: C:\Users\HEESU\AppData\Local\Temp\tmp8ybtdvvm


In [8]:
# memory 파라미터로 AGENTS.md 로드
memory_inject_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 코딩 어시스턴트입니다.",
    backend=FilesystemBackend(root_dir=tmp_dir),
    memory=["/memory/AGENTS.md"],  # AGENTS.md 파일 경로
)

# 에이전트가 AGENTS.md의 규칙을 따르는지 확인
result = memory_inject_agent.invoke(
    {"messages": [{"role": "user", "content": "간단한 HTTP 클라이언트 클래스를 만들어 주세요. 프로젝트 컨벤션을 따라주세요."}]},
    config=lf_config,
)

print(result["messages"][-1].content)

C:\Users\HEESU\AppData\Local\Temp\ipykernel_49928\1142991349.py:5: DeprecationWarning: FilesystemBackend virtual_mode default will change in deepagents 0.5.0; please specify virtual_mode explicitly. Note: virtual_mode is for virtual path semantics (e.g., CompositeBackend routing) and optional path-based guardrails; it does not provide sandboxing or process isolation. Security note: leaving virtual_mode=False allows absolute paths and '..' to bypass root_dir. Consult the API reference for details.
  backend=FilesystemBackend(root_dir=tmp_dir),


이미 /workspace/C:\workspace\http_client.py에 간단한 HTTP 클라이언트 클래스(SimpleHttpClient)가 구현되어 있습니다.  
이 클래스는 프로젝트 컨벤션을 따르며, GET과 POST 요청 모두를 지원합니다.

특정한 추가 사항이나 규칙이 있다면 말씀해 주세요.  
아니면 이 구현에서 수정하거나 보완할 부분이 있다면 알려주시면 반영하겠습니다.


---
## 4. 스킬 (Skills)

스킬은 에이전트에게 **도메인 전문 지식**을 부여하는 모듈화된 지침 세트입니다.

### Memory vs Skills

| 비교 | Memory (AGENTS.md) | Skills (SKILL.md) |
|------|-------------------|-------------------|
| **로딩** | 항상 로드 (Always) | 필요 시 로드 (On-demand) |
| **파일 형식** | `AGENTS.md` | `SKILL.md` (YAML 프론트매터) |
| **적합한 용도** | 항상 적용되는 규칙/컨벤션 | 특정 태스크에 필요한 큰 컨텍스트 |
| **토큰 효율** | 항상 소비 | 점진적 공개로 절약 |
| **크기** | 간결할수록 좋음 | 대용량 가능 (10MB 제한) |
| **업데이트** | 에이전트가 edit_file로 수정 가능 | 보통 정적 |

### Progressive Disclosure (점진적 공개)

스킬은 한 번에 전부 로드하지 않습니다:
1. 처음에는 **프론트매터(이름, 설명)**만 로드
2. 사용자 요청과 관련된 스킬을 **에이전트가 판단**
3. 필요한 스킬의 **전체 내용**을 그때 로드

이 방식으로 토큰을 절약하면서도 필요한 전문 지식에 접근할 수 있습니다.

### SKILL.md 구조

```yaml
---
name: web-research           # 스킬 식별자 (최대 64자, 소문자+하이픈)
description: >               # 설명 (최대 1024자) — 매칭에 사용
  체계적인 웹 리서치를 수행하기 위한 단계별 가이드.
  정보 수집, 검증, 정리까지의 전체 워크플로를 다룹니다.
license: MIT
compatibility: Python 3.8+
metadata:
  category: research
allowed-tools: ls read_file write_file
---

# Web Research 스킬

## 사용 시기
- 사용자가 특정 주제에 대한 조사를 요청할 때
- 최신 정보가 필요한 질문이 들어올 때

## 워크플로
1. 검색 쿼리 설계
2. 다양한 소스에서 정보 수집
3. 정보 교차 검증
4. 구조화된 보고서 작성
```

In [9]:
# 스킬을 사용하는 에이전트 생성
skilled_agent = create_deep_agent(
    model=model,
    system_prompt="당신은 시니어 개발자입니다. 사용 가능한 스킬을 활용하여 작업을 수행하세요.",
    backend=FilesystemBackend(root_dir=tmp_dir),
    skills=["/skills/"],  # 스킬 소스 디렉토리
)

print("스킬 에이전트 생성 완료")

스킬 에이전트 생성 완료


C:\Users\HEESU\AppData\Local\Temp\ipykernel_49928\88414173.py:5: DeprecationWarning: FilesystemBackend virtual_mode default will change in deepagents 0.5.0; please specify virtual_mode explicitly. Note: virtual_mode is for virtual path semantics (e.g., CompositeBackend routing) and optional path-based guardrails; it does not provide sandboxing or process isolation. Security note: leaving virtual_mode=False allows absolute paths and '..' to bypass root_dir. Consult the API reference for details.
  backend=FilesystemBackend(root_dir=tmp_dir),


In [10]:
# 스킬을 활용한 코드 리뷰 요청
result = skilled_agent.invoke(
    {"messages": [{"role": "user", "content": """
다음 코드를 리뷰해 주세요:

```python
import os

def process_data(data):
    result = []
    for item in data:
        if item['type'] == 'user':
            name = item['name']
            email = item['email']
            query = f"INSERT INTO users VALUES ('{name}', '{email}')"
            db.execute(query)
            result.append(item)
    return result
```
"""}]},
    config=lf_config,
)

print(result["messages"][-1].content)

코드 리뷰:

### 1. SQL Injection 취약점
현재 쿼리문:
```python
query = f"INSERT INTO users VALUES ('{name}', '{email}')"
db.execute(query)
```
- 사용자 입력값 name, email이 직접 SQL에 삽입되고 있습니다.
- 이 방식은 SQL Injection에 매우 취약합니다.
- 바람직한 방법은 SQL 파라미터 바인딩(prepared statement) 사용입니다.

예시 (SQLite/Python 기준):
```python
db.execute("INSERT INTO users (name, email) VALUES (?, ?)", (name, email))
```

### 2. db 객체의 정의 부재
- 코드 내에서 db가 어디서 오는지 알 수 없습니다.
- db 객체가 함수에 명시적으로 주입되야 테스트도 쉽고 명확합니다.

### 3. 테이블 컬럼 명시 권장
- 쿼리에서 VALUES 바로 뒤에 컬럼명을 명시하지 않으면, 테이블 컬럼 순서가 바뀔 때 데이터가 잘못 들어갈 수 있음.

### 4. 예외 처리 없음
- 데이터베이스 실행 중 발생하는 예외를 적절히 처리하지 않고 있습니다.

### 5. 데이터 검증/유효성 검사 없음
- name, email 값에 대한 최소한의 유효성 검사가 필요합니다.

### 6. 불필요한 import
- os 모듈을 임포트했지만 사용하지 않았으니 제거해야 합니다.

---

## 개선된 코드 예시

```python
def process_data(data, db):
    result = []
    for item in data:
        if item.get('type') == 'user':
            name = item.get('name')
            email = item.get('email')
            # 간단한 유효성 검사
            if not name or not email

---
## 5. 스킬 소스 우선순위

여러 스킬 소스를 지정하면, **나중에 나온 소스가 우선**합니다 (last wins).

```python
skills=[
    "/skills/base/",     # 기본 스킬
    "/skills/user/",     # 사용자 스킬 (base 덮어쓰기 가능)
    "/skills/project/",  # 프로젝트 스킬 (최우선)
]
```

같은 이름의 스킬이 여러 소스에 있으면, 마지막 소스의 스킬이 사용됩니다.

---
## 6. 서브에이전트의 스킬 상속

| 서브에이전트 유형 | 스킬 상속 |
|-------------------|----------|
| General-purpose (빌트인) | 메인 에이전트의 스킬을 **자동 상속** |
| 커스텀 SubAgent | **명시적 `skills` 파라미터** 필요 |

```python
# 커스텀 서브에이전트에 스킬 부여
subagent = {
    "name": "reviewer",
    "description": "코드 리뷰 전문 에이전트",
    "system_prompt": "...",
    "tools": [],
    "skills": ["/skills/code-review/"],  # 명시적 스킬 경로
}
```

---
## 핵심 정리

| 항목 | 내용 |
|------|------|
| 장기 메모리 | `CompositeBackend` + `StoreBackend`로 `/memories/` 영속화 |
| AGENTS.md | `memory=["/path/AGENTS.md"]` → 항상 시스템 프롬프트에 주입 |
| Skills | `skills=["/skills/"]` → SKILL.md 기반 점진적 공개 |
| Progressive Disclosure | 프론트매터만 먼저 로드 → 필요 시 전체 로드 |
| 스킬 우선순위 | 나중 소스가 우선 (last wins) |
| Memory vs Skills | Memory = 항상 로드 / Skills = 필요 시 로드 |

## 다음 단계
→ **[07_advanced.ipynb](./07_advanced.ipynb)**: Human-in-the-Loop, 스트리밍, 샌드박스 등 고급 기능을 배웁니다.